### Feature extraction in the Weather-Image Dataset

This Notebook sticks to do the required computations. 

For a description of this proccess, see **Section Preprocessing** in `main.ipynb` in the same folder as this Notebook.

**Requirements:**

The `R` package `magick` (which depends on `Rcpp`) is required to process the images (eg. to extract RGB channels). The following `R` packages were installed as follows on JupyterLab:

In [5]:
library(future)
library(future.apply)
library(progressr)
library(magick)
packageVersion("magick")

source("./fn-img-tools.R")

[1] ‘2.9.1’

In [1]:
unzip("./archive_weather.zip", exdir = "original-img-weather-dataset")

In [6]:
img_paths <- list.files(path = "./original-img-weather-dataset/dataset", pattern = "jpg", #"\\.(jpg|jpeg)$",
  recursive = TRUE, full.names = TRUE, ignore.case = TRUE)
print(img_paths[1:4])

[1] "./original-img-weather-dataset/dataset/dew/2208.jpg"
[2] "./original-img-weather-dataset/dataset/dew/2209.jpg"
[3] "./original-img-weather-dataset/dataset/dew/2210.jpg"
[4] "./original-img-weather-dataset/dataset/dew/2211.jpg"


In [9]:
test_run <- extract_rgb_channels(img_paths[1])
#test_run

In [11]:
test_run <- compute_rgb_histogram_features(img_paths[1])
#test_run

In [12]:
available_workers <- parallel::detectCores()
nworkers <- 50 #parallel::detectCores() - 1
cat(paste(sprintf("Using %d cores out of %d available cores.", nworkers, available_workers)))

Using 50 cores out of 256 available cores.

In [14]:
# Parallel processing of images.

future::plan( future::multisession, workers = nworkers)

progressr::handlers("txtprogressbar")

niter <- length(img_paths)

res_list <- progressr::with_progress({
  p <- progressr::progressor(along = seq_len(niter))

  future.apply::future_lapply(
    X = seq_len(niter),
    FUN = function(i) {
      res <- compute_rgb_histogram_features(img_paths[[i]])
      p() # update progress
      res
    },
    future.seed = NULL
  )
})

future::plan(future::sequential) # back to sequential processing

In [15]:
n_of_features <- length(res_list[[1]]$features) # 3 x 32
all_imgs_features <- matrix(NA, nrow = length(img_paths), ncol = n_of_features,
  dimnames = list(
    paste0(basename(dirname(img_paths)), "-", tools::file_path_sans_ext(basename(img_paths))),
    paste0("features", seq_len(n_of_features))))

id_fail <- integer(0)

for (i in seq_along(res_list))
{
  if (is.list(res_list[[i]]) && !is.null(res_list[[i]]$err) && !res_list[[i]]$err) {
    all_imgs_features[i,] <- res_list[[i]]$features
  } else {
    id_fail <- c(id_fail, i)
  }

  # the following may generate message: IOPub message rate exceeded.
  #cat(sprintf("\rProcessed %d / %d", i, length(res_list)))
  #flush.console()
}
# Processed 6862 / 6862

id_ok <- setdiff(seq_along(img_paths), id_fail)

cat("Correctly processed images:", length(id_ok), "\n")
cat("Failed images:", length(id_fail), "\n")
# Correctly processed images: 6860
# Failed images: 2

print(img_paths[id_fail])

Correctly processed images: 6860 
Failed images: 2 
[1] "./original-img-weather-dataset/dataset/fogsmog/4514.jpg"
[2] "./original-img-weather-dataset/dataset/snow/1187.jpg"   


In [16]:
for (i in id_fail)
{
  print(res_list[[i]]$msg)
  header <- readBin(img_paths[i], what = "raw", n = 6)
  #print(identical(header, charToRaw("GIF87a")) || identical(header, charToRaw("GIF89a")))
  cat(paste(res_list[[i]]$msg, ":", rawToChar(header)))
}

[1] "read error"
read error : GIF89a[1] "read error"
read error : GIF89a

In [17]:
labels_ok <- basename(dirname(img_paths))[id_ok]

X_hist <- all_imgs_features[id_ok,,drop=FALSE]

# See use this scaled version, but normalized X_hist seems okay.
X_hist_scaled <- scale(X_hist)

saveRDS(list(X_hist = X_hist, X_hist_scaled = X_hist_scaled,
  labels = labels_ok, img_paths = img_paths[id_ok], id_fail = id_fail),
  file = "rgb_histograms_32bins.rds")

In [18]:
# remove the original images directory to save space.
# To see the images unzip archive_weather.zip and browse it or 
# browse img-weather-thumbs-150x100, which contains 
# smaller thumbnails used to create the collages for each cluster.
unlink("./original-img-weather-dataset", recursive = TRUE, force = TRUE)